<a href="https://colab.research.google.com/github/ravipallisaisarath/Technology_and_Data_Management_Case_Study/blob/main/Technology_and_Data_Management_Case_Study_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np
import re
from google.colab import files

uploaded = files.upload()
file_path = next(iter(uploaded))

print("File uploaded:", file_path)

Saving Case_Data_Technology_and_Data_Management_All_Sheets.xlsx to Case_Data_Technology_and_Data_Management_All_Sheets.xlsx
File uploaded: Case_Data_Technology_and_Data_Management_All_Sheets.xlsx


In [20]:
items = pd.read_excel(file_path, sheet_name="Item_Master")
stock = pd.read_excel(file_path, sheet_name="Stock_Aging")
sales = pd.read_excel(file_path, sheet_name="Sales_Lines")
customers = pd.read_excel(file_path, sheet_name="Customers")

for df in [items, stock, sales, customers]:
    df.columns = [str(c).strip() for c in df.columns]

print("All worksheets loaded successfully.")
print("Items:", items.shape)
print("Stock:", stock.shape)
print("Sales:", sales.shape)
print("Customers:", customers.shape)

All worksheets loaded successfully.
Items: (42, 8)
Stock: (42, 6)
Sales: (630, 7)
Customers: (30, 4)


Load all worksheets

In [21]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def normalize_text(x):
    return clean_text(x).casefold()

def clean_sku(x):
    return clean_text(x).upper()

def clean_id(x):
    if pd.isna(x):
        return ""
    if isinstance(x, (int, np.integer)):
        return str(x)
    if isinstance(x, (float, np.floating)) and x.is_integer():
        return str(int(x))
    value = clean_text(x)
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]
    return value

for df in [items, stock, sales]:
    df["SKU"] = df["SKU"].apply(clean_sku)

for df in [sales, customers]:
    df["CustomerID"] = df["CustomerID"].apply(clean_id)

items["Status_clean"] = items["Status"].apply(normalize_text)
items["Flag_clean"] = items["Alternate Flag"].apply(
    lambda x: clean_text(x).upper()
)

for col in ["Master Pack Qty", "Unit Cost (Rs)"]:
    items[col] = pd.to_numeric(items[col], errors="coerce").fillna(0)

stock_buckets = [
    "Qty 0-30 days",
    "Qty 31-60 days",
    "Qty 61-90 days",
    "Qty 91-180 days",
    "Qty 180+ days"
]

for col in stock_buckets:
    stock[col] = pd.to_numeric(stock[col], errors="coerce").fillna(0)

for col in ["Qty", "Value (Rs)"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce").fillna(0)

sales["Month"] = pd.to_datetime(sales["Month"], errors="coerce")

print("Data cleaned successfully.")

Data cleaned successfully.


/tmp/ipykernel_1487/771975418.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales["Month"] = pd.to_datetime(sales["Month"], errors="coerce")


Clean data

In [22]:
name_to_sku = {}

for _, row in items.iterrows():
    name = normalize_text(row["Item Name"])
    if name:
        name_to_sku[name] = row["SKU"]

switch_map = {}
switch_issues = []

for _, row in items.iterrows():
    if row["Flag_clean"] == "SWITCH":
        old_sku = row["SKU"]
        alternate_name = normalize_text(row["Alternate Item (name)"])
        new_sku = name_to_sku.get(alternate_name)

        if not new_sku:
            switch_issues.append({
                "Issue": "Replacement item not found",
                "SKU": old_sku,
                "Details": row["Alternate Item (name)"]
            })
        elif new_sku == old_sku:
            switch_issues.append({
                "Issue": "Replacement points to the same SKU",
                "SKU": old_sku,
                "Details": alternate_name
            })
        else:
            switch_map[old_sku] = new_sku

def canonical_sku(sku):
    sku = clean_sku(sku)
    visited = set()

    while sku in switch_map:
        if sku in visited:
            raise ValueError(f"SWITCH mapping cycle detected: {sku}")
        visited.add(sku)
        sku = switch_map[sku]

    return sku

stock["Reporting SKU"] = stock["SKU"].apply(canonical_sku)
sales["Reporting SKU"] = sales["SKU"].apply(canonical_sku)

print("SWITCH mapping:", switch_map)
print("Unresolved SWITCH issues:", len(switch_issues))

SWITCH mapping: {'SKU1003': 'SKU1006', 'SKU1015': 'SKU1019', 'SKU1030': 'SKU1033'}
Unresolved SWITCH issues: 0


Identify SWITCH items

In [23]:
items["Is_Inactive"] = items["Status_clean"].str.contains(
    "inactive", na=False
)

original_stock = stock.groupby("SKU")[stock_buckets].sum()

inactive_stock_issues = []

for sku in items.loc[items["Is_Inactive"], "SKU"].unique():
    if sku in original_stock.index:
        quantity = original_stock.loc[sku, stock_buckets].sum()

        if quantity > 0:
            inactive_stock_issues.append({
                "Issue": "Inactive item has stock",
                "SKU": sku,
                "Details": f"Stock quantity: {quantity}"
            })

print("Inactive items with stock:", len(inactive_stock_issues))
display(pd.DataFrame(inactive_stock_issues))

Inactive items with stock: 1


,Issue,SKU,Details
0,Inactive item has stock,SKU1041,Stock quantity: 169


Flag inactive items with stock

In [24]:
stock_agg = (
    stock.groupby("Reporting SKU")[stock_buckets]
    .sum()
    .reset_index()
    .rename(columns={"Reporting SKU": "SKU"})
)

# April to September 2026
sales_6m = sales[
    sales["Month"].ge(pd.Timestamp("2026-04-01"))
    & sales["Month"].lt(pd.Timestamp("2026-10-01"))
].copy()

sales_agg = (
    sales_6m.groupby("Reporting SKU")
    .agg(
        Sales_Qty_6M=("Qty", "sum"),
        Sales_Value_6M=("Value (Rs)", "sum")
    )
    .reset_index()
    .rename(columns={"Reporting SKU": "SKU"})
)

active_items = items[~items["Is_Inactive"]].copy()
active_items = active_items.drop_duplicates("SKU", keep="first")

analysis = active_items.merge(
    stock_agg, on="SKU", how="left"
).merge(
    sales_agg, on="SKU", how="left"
)

for col in stock_buckets + ["Sales_Qty_6M", "Sales_Value_6M"]:
    analysis[col] = analysis[col].fillna(0)

analysis["Total Stock Qty"] = analysis[stock_buckets].sum(axis=1)

analysis["Aged Stock Qty"] = (
    analysis["Qty 91-180 days"] + analysis["Qty 180+ days"]
)

analysis["Aged Stock Value"] = (
    analysis["Aged Stock Qty"] * analysis["Unit Cost (Rs)"]
)

analysis["Average Monthly Sales"] = analysis["Sales_Qty_6M"] / 6

analysis["Months to Clear"] = np.where(
    analysis["Average Monthly Sales"] > 0,
    analysis["Total Stock Qty"] / analysis["Average Monthly Sales"],
    np.where(analysis["Total Stock Qty"] > 0, np.inf, 0)
)

print("Stock and sales aggregated.")
display(analysis.head())

Stock and sales aggregated.


,SKU,Item Name,Category,Master Pack Qty,Unit Cost (Rs),Alternate Item (name),Alternate Flag,Status,Status_clean,Flag_clean,...,Qty 61-90 days,Qty 91-180 days,Qty 180+ days,Sales_Qty_6M,Sales_Value_6M,Total Stock Qty,Aged Stock Qty,Aged Stock Value,Average Monthly Sales,Months to Clear
0,SKU1001,A-Product 01 (50/case),Category A,50,25,NaN,NaN,Active,active,,...,75.0,20.0,60.0,939.0,59874.0,430.0,80.0,2000.0,156.500000,2.747604
1,SKU1002,A-Product 02 (10/case),Category A,10,85,NaN,NaN,Active,active,,...,6.0,7.0,4.0,3.0,591.0,37.0,11.0,935.0,0.500000,74.000000
2,SKU1003,A-Product 03 (12/case),Category A,12,60,A-Product 06 (10/case),SWITCH,Active,active,SWITCH,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
3,SKU1004,A-Product 04 (12/case),Category A,12,25,NaN,NaN,Active,active,,...,14.0,0.0,0.0,674.0,40327.0,96.0,0.0,0.0,112.333333,0.854599
4,SKU1005,A-Product 05 (50/case),Category A,50,340,NaN,NaN,Active,active,,...,46.0,55.0,37.0,280.0,471138.0,306.0,92.0,31280.0,46.666667,6.557143


Aggregate stock and sales

In [25]:
def classify_item(row):
    # R5: Apply the rules in the specified order
    if row["Sales_Qty_6M"] < row["Master Pack Qty"]:
        return "Red List"

    if (
        row["Aged Stock Qty"] == 0
        or row["Months to Clear"] < 3
    ):
        return "No Action"

    return "Push"

analysis["Classification"] = analysis.apply(
    classify_item, axis=1
)

classification_counts = (
    analysis["Classification"]
    .value_counts()
    .reindex(["Red List", "No Action", "Push"], fill_value=0)
    .rename_axis("Classification")
    .reset_index(name="Item Count")
)

display(classification_counts)

,Classification,Item Count
0,Red List,10
1,No Action,13
2,Push,18


Classify items

In [26]:
push_items = analysis[
    analysis["Classification"] == "Push"
].copy()

top_10_push = push_items.sort_values(
    "Aged Stock Value", ascending=False
).head(10)

print("Top 10 Push items:")
display(top_10_push[[
    "SKU", "Item Name", "Category",
    "Aged Stock Qty", "Aged Stock Value",
    "Sales_Qty_6M", "Total Stock Qty", "Months to Clear"
]])

total_push_aged_value = push_items["Aged Stock Value"].sum()

print(f"Total aged stock value for Push items: Rs {total_push_aged_value:,.2f}")

Top 10 Push items:


,SKU,Item Name,Category,Aged Stock Qty,Aged Stock Value,Sales_Qty_6M,Total Stock Qty,Months to Clear
4,SKU1005,A-Product 05 (50/case),Category A,92.0,31280.0,280.0,306.0,6.557143
33,SKU1034,E-Product 06 (100/case),Category E,239.0,20315.0,581.0,797.0,8.230637
19,SKU1020,C-Product 06 (100/case),Category C,153.0,18360.0,479.0,511.0,6.400835
18,SKU1019,C-Product 05 (12/case),Category C,210.0,12600.0,1368.0,907.0,3.978070
28,SKU1029,E-Product 01 (25/case),Category E,113.0,9605.0,514.0,379.0,4.424125
5,SKU1006,A-Product 06 (10/case),Category A,61.0,9150.0,306.0,203.0,3.980392
32,SKU1033,E-Product 05 (10/case),Category E,137.0,8220.0,861.0,455.0,3.170732
13,SKU1014,B-Product 07 (24/case),Category B,90.0,7650.0,555.0,301.0,3.254054
8,SKU1009,B-Product 02 (12/case),Category B,22.0,7480.0,93.0,72.0,4.645161
37,SKU1038,F-Product 03 (24/case),Category F,61.0,7320.0,171.0,204.0,7.157895


Total aged stock value for Push items: Rs 163,841.00


Top 10 Push items and total aged stock value

In [27]:
requested_skus = ["SKU1001", "SKU1006", "SKU1019"]

sku_results = analysis[
    analysis["SKU"].isin(requested_skus)
][[
    "SKU", "Item Name", "Classification",
    "Sales_Qty_6M", "Master Pack Qty",
    "Total Stock Qty", "Aged Stock Qty",
    "Aged Stock Value", "Months to Clear"
]]

display(sku_results)

missing_skus = sorted(set(requested_skus) - set(analysis["SKU"]))
print("Requested SKUs not found in active analysis:", missing_skus)

,SKU,Item Name,Classification,Sales_Qty_6M,Master Pack Qty,Total Stock Qty,Aged Stock Qty,Aged Stock Value,Months to Clear
0,SKU1001,A-Product 01 (50/case),No Action,939.0,50,430.0,80.0,2000.0,2.747604
5,SKU1006,A-Product 06 (10/case),Push,306.0,10,203.0,61.0,9150.0,3.980392
18,SKU1019,C-Product 05 (12/case),Push,1368.0,12,907.0,210.0,12600.0,3.978070


Requested SKUs not found in active analysis: []


Requested SKU classifications

In [28]:
customers["Industry_clean"] = customers["Industry"].apply(normalize_text)

customer_totals = (
    sales_6m.groupby("CustomerID", as_index=False)
    .agg(Customer_Total_Value=("Value (Rs)", "sum"))
)

customer_info = customers.drop_duplicates(
    "CustomerID", keep="first"
).merge(
    customer_totals, on="CustomerID", how="left"
)

customer_info["Customer_Total_Value"] = (
    customer_info["Customer_Total_Value"].fillna(0)
)

eligible_customers = customer_info[
    customer_info["Customer_Total_Value"] > 100000
].copy()

customer_sales = sales_6m.merge(
    customers[["CustomerID", "Industry", "Area"]].drop_duplicates("CustomerID"),
    on="CustomerID",
    how="left",
    suffixes=("", "_customer")
)

customer_sales["Industry_clean"] = customer_sales["Industry"].apply(
    normalize_text
)

sku_category = (
    active_items.drop_duplicates("SKU")
    .set_index("SKU")["Category"].to_dict()
)

sku_name = (
    active_items.drop_duplicates("SKU")
    .set_index("SKU")["Item Name"].to_dict()
)

buyers_by_sku = (
    customer_sales.groupby("Reporting SKU")["CustomerID"]
    .apply(set).to_dict()
)

industries_by_sku = (
    customer_sales.groupby("Reporting SKU")["Industry_clean"]
    .apply(lambda s: set(x for x in s if x))
    .to_dict()
)

categories_by_customer = (
    customer_sales.assign(
        Purchased_Category=customer_sales["Reporting SKU"].map(sku_category)
    )
    .groupby("CustomerID")["Purchased_Category"]
    .apply(lambda s: set(normalize_text(x) for x in s if normalize_text(x)))
    .to_dict()
)

recommendations = []

for _, item in push_items.iterrows():
    sku = item["SKU"]
    category = normalize_text(item["Category"])

    buyers = buyers_by_sku.get(sku, set())
    buyer_industries = industries_by_sku.get(sku, set())

    for _, cust in eligible_customers.iterrows():
        cid = cust["CustomerID"]
        industry = cust["Industry_clean"]

        if cid in buyers:
            bucket = 1
        elif industry and industry in buyer_industries:
            bucket = 2
        elif category and category in categories_by_customer.get(cid, set()):
            bucket = 3
        else:
            continue

        recommendations.append({
            "SKU": sku,
            "Item Name": sku_name.get(sku, ""),
            "CustomerID": cid,
            "CustomerName": cust["CustomerName"],
            "Industry": cust["Industry"],
            "Area": cust["Area"],
            "Customer Total Purchase Value": cust["Customer_Total_Value"],
            "Bucket": bucket
        })

recommendations_df = pd.DataFrame(
    recommendations,
    columns=[
        "SKU", "Item Name", "CustomerID", "CustomerName",
        "Industry", "Area", "Customer Total Purchase Value", "Bucket"
    ]
)

if not recommendations_df.empty:
    recommendations_df = recommendations_df.sort_values(
        ["SKU", "Bucket", "Customer Total Purchase Value"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

display(recommendations_df.head(20))

,SKU,Item Name,CustomerID,CustomerName,Industry,Area,Customer Total Purchase Value,Bucket
0,SKU1005,A-Product 05 (50/case),C220,Customer 220,Caterer,Zone 2,689088.0,1
1,SKU1005,A-Product 05 (50/case),C224,Customer 224,Restaurant,Zone 1,320780.0,1
2,SKU1005,A-Product 05 (50/case),C214,Customer 214,Caterer,Zone 1,574001.0,2
3,SKU1005,A-Product 05 (50/case),C208,Customer 208,Caterer,Zone 2,502556.0,2
4,SKU1005,A-Product 05 (50/case),C202,Customer 202,Caterer,Zone 3,428435.0,2
5,SKU1005,A-Product 05 (50/case),C206,Customer 206,Restaurant,Zone 1,253883.0,2
6,SKU1005,A-Product 05 (50/case),C226,Customer 226,Caterer,Zone 3,106498.0,2
7,SKU1005,A-Product 05 (50/case),C221,Customer 221,Hotel,Zone 2,209491.0,3
8,SKU1005,A-Product 05 (50/case),C227,Customer 227,Hotel,Zone 2,182264.0,3
9,SKU1006,A-Product 06 (10/case),C208,Customer 208,Caterer,Zone 2,502556.0,1


Customer recommendations

In [29]:
sku1005_customers = recommendations_df[
    recommendations_df["SKU"] == "SKU1005"
].head(3)

display(sku1005_customers)

if sku1005_customers.empty:
    print("No eligible customers matched the rules for SKU1005.")

,SKU,Item Name,CustomerID,CustomerName,Industry,Area,Customer Total Purchase Value,Bucket
0,SKU1005,A-Product 05 (50/case),C220,Customer 220,Caterer,Zone 2,689088.0,1
1,SKU1005,A-Product 05 (50/case),C224,Customer 224,Restaurant,Zone 1,320780.0,1
2,SKU1005,A-Product 05 (50/case),C214,Customer 214,Caterer,Zone 1,574001.0,2


First three customers for SKU1005

In [30]:
quality_issues = list(switch_issues) + list(inactive_stock_issues)

# Duplicate master SKUs
for sku in sorted(items.loc[
    items.duplicated("SKU", keep=False), "SKU"
].unique()):
    quality_issues.append({
        "Issue": "Duplicate SKU in Item_Master",
        "SKU": sku,
        "Details": ""
    })

known_skus = set(items["SKU"])

for sku in sorted(set(stock["SKU"]) - known_skus):
    quality_issues.append({
        "Issue": "Stock SKU missing from Item_Master",
        "SKU": sku,
        "Details": ""
    })

for sku in sorted(set(sales["SKU"]) - known_skus):
    quality_issues.append({
        "Issue": "Sales SKU missing from Item_Master",
        "SKU": sku,
        "Details": ""
    })

invalid_months = int(sales["Month"].isna().sum())

if invalid_months:
    quality_issues.append({
        "Issue": "Missing or invalid sales month",
        "SKU": "",
        "Details": f"{invalid_months} rows"
    })

quality_df = pd.DataFrame(
    quality_issues,
    columns=["Issue", "SKU", "Details"]
).drop_duplicates().reset_index(drop=True)

display(quality_df)

,Issue,SKU,Details
0,Inactive item has stock,SKU1041,Stock quantity: 169


Data quality checks

In [31]:
output_file = "Case_Study_Results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    analysis.to_excel(writer, sheet_name="Item Analysis", index=False)
    classification_counts.to_excel(writer, sheet_name="Class Counts", index=False)
    top_10_push.to_excel(writer, sheet_name="Top 10 Push", index=False)
    sku_results.to_excel(writer, sheet_name="Requested SKUs", index=False)
    recommendations_df.to_excel(
        writer, sheet_name="Customer Recommendations", index=False
    )
    sku1005_customers.to_excel(
        writer, sheet_name="SKU1005 Top 3", index=False
    )
    quality_df.to_excel(
        writer, sheet_name="Data Quality Issues", index=False
    )

print("Excel report created successfully:", output_file)

files.download(output_file)

Excel report created successfully: Case_Study_Results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Export all results to Excel